In [54]:
using Pkg                # Load Julia's built-in package manager module (Pkg)

Pkg.activate("..")       # Activate the Project.toml located one directory above
                         # (Allows notebooks/ to use the repository root environment)

using Revise             # Development tool that automatically reloads changes
                         # made to files in src/ without restarting Julia

using Horizons3DViewer2  # Load the custom package Horizons3DViewer2
                         # (This loads src/Horizons3DViewer2.jl and its modules)

using EphemerisSources   # Tools for accessing planetary ephemeris data
using AstroTime, Dates   # Time systems and date handling utilities
using GLMakie            # High-performance interactive visualization backend
using ProgressMeter      # Progress bar utilities for long computations
using LinearAlgebra      # Standard linear algebra operations (norm, dot, etc.)

  Activating project at `~/work/Horizons3DViewer2`


https://juliaastro.org/EphemerisSources.jl/docs/stable/examples/telescope-study/
https://juliaastro.org/AstroTime/stable/tutorial/#Tutorial
https://ssd.jpl.nasa.gov/horizons/app.html#/

In [55]:
######################################
# Specify the data to be retrieved
######################################

frame = "J2000"   # Input parameter for the NASA Horizons API
wrt   = "sun"     # Input parameter for the NASA Horizons API
units = "KM-S"    # Input parameter for the NASA Horizons API

####################
spacecraft = "Voyager 1"
start = TDBEpoch(1977, 09, 6, 0, 0, 0.0)
stop  = TDBEpoch(2000, 07, 25, 0, 0, 0.0)
step  = "3 day"
step_size_sec = 3*24*60*60


259200

In [56]:
############################################################################
# Retrieve ephemeris data and convert it into a convenient internal format
############################################################################

# Retrieve ephemeris data from JPL Horizons
sun_horizons     = ephemeris("sun", start, stop, step; wrt = wrt, units = units, frame = frame)
mercury_horizons = ephemeris("mercury", start, stop, step; wrt = wrt, units = units, frame = frame)
venus_horizons   = ephemeris("venus", start, stop, step; wrt = wrt, units = units, frame = frame)
earth_horizons   = ephemeris("earth", start, stop, step; wrt = wrt, units = units, frame = frame)
mars_horizons    = ephemeris("mars", start, stop, step; wrt = wrt, units = units, frame = frame)
jupiter_horizons = ephemeris("jupiter", start, stop, step; wrt = wrt, units = units, frame = frame)
saturn_horizons  = ephemeris("saturn", start, stop, step; wrt = wrt, units = units, frame = frame)
uranus_horizons  = ephemeris("uranus", start, stop, step; wrt = wrt, units = units, frame = frame)
neptune_horizons = ephemeris("neptune", start, stop, step; wrt = wrt, units = units, frame = frame)
pluto_horizons   = ephemeris("999", start, stop, step; wrt = wrt, units = units, frame = frame)

sc1_horizons     = ephemeris("Voyager 1", start, stop, step; wrt = wrt, units = units, frame = frame)
sc2_horizons     = ephemeris("Voyager 2", start, stop, step; wrt = wrt, units = units, frame = frame)

# Convert the Horizons outputs into 6×N state matrices
sun_states       = horizons_to_matrix(sun_horizons)
mercury_states   = horizons_to_matrix(mercury_horizons)
venus_states     = horizons_to_matrix(venus_horizons)
earth_states     = horizons_to_matrix(earth_horizons)
mars_states      = horizons_to_matrix(mars_horizons)
jupiter_states   = horizons_to_matrix(jupiter_horizons)
saturn_states    = horizons_to_matrix(saturn_horizons)
uranus_states    = horizons_to_matrix(uranus_horizons)
neptune_states   = horizons_to_matrix(neptune_horizons)
pluto_states     = horizons_to_matrix(pluto_horizons)

sc1_states       = horizons_to_matrix(sc1_horizons)
sc2_states       = horizons_to_matrix(sc2_horizons)

# Convert Horizons time stamps (Julian Date) to TDBEpoch values for animation labels
t_tdb = TDBEpoch.(sc1_horizons.t .* days; origin = :julian)

# Diagnostic output (for sanity checks)
println(size(sc1_states))
println(sc1_states[1:6, 1])
println(t_tdb[1])

(6, 2787)
[1.448005439108036e8, -4.220149282221936e7, 25598.45101304352, 11.57190972543096, 38.13748393606272, 0.7305166962567302]
1977-09-06T00:00:00.000 TDB


In [58]:
######################################
# Animation settings
######################################

# Whether to render the animation (0: static image, 1: animation)
animation_flag = 1

# Length of the spacecraft trajectory trail
trail_len = 100000

# Animation frame selection
sampling_interval = 5
N = min(size(sc1_states, 2), size(sc2_states, 2), size(earth_states, 2), size(mars_states, 2), size(jupiter_states, 2), size(saturn_states, 2), size(uranus_states, 2), size(neptune_states, 2), length(t_tdb))
frames = 1:sampling_interval:(N-1)

# Zoom settings (same units as the state coordinates: km)
zoom_min = 50e7
zoom_margin = 0.50

# Required half-width based on the distance of Voyager 1 and 2 from the Sun
zoom_radius1 = [sqrt(sum(abs2, sc1_states[1:3, i])) for i in 1:N]
zoom_radius2 = [sqrt(sum(abs2, sc2_states[1:3, i])) for i in 1:N]

# Never shrink the viewing range once expanded
zoom_extent = accumulate(max, max.(zoom_min, zoom_margin .* max.(zoom_radius1, zoom_radius2)))

######################################
# Plotting area settings
######################################

set_theme!(theme_black())
update_theme!(fontsize=18)

fig1 = Figure(size=(900, 600))
ax_ecliptic_s = default_axis3(fig1[1, 1]; scale=zoom_min, title="")

ax_ecliptic_s.aspect = :data
ax_ecliptic_s.viewmode = :fit

######################################
# Common plotting routines
######################################

# Sun
scatter!(ax_ecliptic_s, 0.0, 0.0, 0.0; markersize=15, color=:white)
text!(ax_ecliptic_s, 0.0, 0.0, 0.0; text="Sun", color=:white, fontsize=18, offset=(8, 8))

# Planet data and label colors
planet_data = [
    (name="Earth",   states=earth_states,   color=:white, trail=trail_len_short),
    (name="Mars",    states=mars_states,    color=:white, trail=trail_len_long),
    (name="Jupiter", states=jupiter_states, color=:white, trail=trail_len_long),
    (name="Saturn",  states=saturn_states,  color=:white, trail=trail_len_long),
    (name="Uranus",  states=uranus_states,  color=:white, trail=trail_len_long),
    (name="Neptune", states=neptune_states, color=:white, trail=trail_len_long),
    (name="Pluto", states=pluto_states, color=:white, trail=trail_len_short),
]

######################################
# Render static image or animation
######################################

if animation_flag == 0

    # Planet trajectories and labels at the last frame
    for planet in planet_data
        states = planet.states
        lines!(ax_ecliptic_s, states[1:3, 1:N]; linewidth=1, color=planet.color)
        scatter!(ax_ecliptic_s, [Point3f(states[1, N], states[2, N], states[3, N])]; markersize=10, color=planet.color)
        text!(ax_ecliptic_s, Point3f(states[1, N], states[2, N], states[3, N]); text=planet.name, color=planet.color, fontsize=18, offset=(8, 8))
    end

    # Voyager trajectories
    lines!(ax_ecliptic_s, sc1_states[1:3, 1:N]; linewidth=1, color=:lime)
    lines!(ax_ecliptic_s, sc2_states[1:3, 1:N]; linewidth=1, color=:orange)

    text!(ax_ecliptic_s, Point3f(sc1_states[1, N], sc1_states[2, N], sc1_states[3, N]); text="Voyager 1", color=:lime, fontsize=18, offset=(8, 8))
    text!(ax_ecliptic_s, Point3f(sc2_states[1, N], sc2_states[2, N], sc2_states[3, N]); text="Voyager 2", color=:orange, fontsize=18, offset=(8, 8))

    # Include all plotted trajectories in the static image
    L = max(zoom_extent[N], zoom_margin * maximum(abs, earth_states[1:3, 1:N]), zoom_margin * maximum(abs, mars_states[1:3, 1:N]), zoom_margin * maximum(abs, jupiter_states[1:3, 1:N]), zoom_margin * maximum(abs, saturn_states[1:3, 1:N]), zoom_margin * maximum(abs, uranus_states[1:3, 1:N]), zoom_margin * maximum(abs, neptune_states[1:3, 1:N]))
    limits!(ax_ecliptic_s, -L, L, -L, L, -L, L)

else

    frame_0 = Observable(first(frames))

    # Initial viewing range
    L0 = zoom_extent[first(frames)]
    limits!(ax_ecliptic_s, -L0, L0, -L0, L0, -L0, L0)

    # Date
    text!(ax_ecliptic_s, 0.95, 0.85; text=@lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative)

    # Planet trajectories and moving labels
    for planet in planet_data
        let states = planet.states, name = planet.name, color = planet.color, trail = planet.trail
            position = @lift(Point3f(states[1, $frame_0], states[2, $frame_0], states[3, $frame_0]))
            text!(ax_ecliptic_s, position; text=name, color=color, fontsize=18, offset=(8, 8))
            traj_fade!(ax_ecliptic_s, states[1:3, :], frame_0; trail_len=trail, color=color, markersize=10)
        end
    end

    # Voyager trajectories
    traj_fade!(ax_ecliptic_s, sc1_states[1:3, :], frame_0; trail_len=trail_len, color=:lime)
    traj_fade!(ax_ecliptic_s, sc2_states[1:3, :], frame_0; trail_len=trail_len, color=:orange)

    # Moving Voyager labels
    text!(ax_ecliptic_s, @lift(Point3f(sc1_states[1, $frame_0], sc1_states[2, $frame_0], sc1_states[3, $frame_0])); text="Voyager 1", color=:lime, fontsize=18, offset=(8, 8))
    text!(ax_ecliptic_s, @lift(Point3f(sc2_states[1, $frame_0], sc2_states[2, $frame_0], sc2_states[3, $frame_0])); text="Voyager 2", color=:orange, fontsize=18, offset=(8, 8))

    p = Progress(length(frames); desc="Rendering movie")

    # Export animation
    record(fig1, "../output/$(spacecraft)_v002.mp4", frames; framerate=30) do frame
        frame_0[] = frame

        # Zoom out as Voyager 1 and 2 move away from the Sun
        L = zoom_extent[frame]
        limits!(ax_ecliptic_s, -L, L, -L, L, -L, L)

        next!(p)
    end

end

display(fig1; inline=false)

Rendering movie 100%|████████████████████████████████████| Time: 0:00:19


GLMakie.Screen(...)